# Constraint satisfaction problems (CSPs)

_Artificial Intelligence (CS221)_

**Fill in variables so all the rules are happy. Many puzzles are exactly this.**

A constraint satisfaction problem asks: assign values to variables so that all the rules hold.

     Each variable picks a value from its domain (its allowed choices).

---

This notebook builds the **anatomy of a CSP** from scratch, one piece at a time — variables, domains, and constraints — and a test that checks whether a complete assignment satisfies every rule. We'll model the Australia map-coloring problem and compare a valid coloring against one that breaks a border rule. Run each cell, read the note above it, then tackle the practice problems at the end. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python

We'll model the **Australia map-coloring** CSP and write a checker that decides whether a given assignment satisfies every constraint. We build it in three steps: (1) the variables, domains, and constraints, (2) the satisfaction test, (3) a good and a deliberately broken assignment.

### Step 1 — Define variables, domains, and constraints

The three ingredients of any CSP: the **variables** (six states), the **domain** of each variable (the three colors it may take), and the **constraints** (border pairs that must differ). Listing all three explicitly is what *modeling* a problem as a CSP means.

In [ ]:
variables = ["WA", "NT", "SA", "Q", "NSW", "V"]

# Every variable shares the same domain of three colors.
domains = {v: ["red", "green", "blue"] for v in variables}

# Each edge is a pair of neighboring states that must differ in color.
edges = [
    ("WA", "NT"), ("WA", "SA"), ("NT", "SA"), ("NT", "Q"),
    ("SA", "Q"), ("SA", "NSW"), ("SA", "V"), ("Q", "NSW"), ("NSW", "V"),
]

### Step 2 — Test whether an assignment satisfies the constraints

An assignment satisfies the CSP only if *every* constraint holds. Here each constraint says the two endpoints of a border must have different colors, so we check that `assign[a] != assign[b]` for all edges. `all(...)` is `True` only when no rule is broken.

In [ ]:
def satisfied(assign):
    # Every border must connect two differently-colored states.
    return all(assign[a] != assign[b] for a, b in edges)

### Step 3 — Try a good assignment and a broken one

We hand-build a valid coloring, then make a copy and recolor `NSW` to blue. Since `SA` is already blue and `SA`–`NSW` is a border, the copy now violates a constraint. Running the checker on both should print `True` then `False`.

In [ ]:
good = {"WA": "red", "NT": "green", "SA": "blue",
        "Q": "red", "NSW": "green", "V": "red"}

bad = dict(good)        # copy the good coloring...
bad["NSW"] = "blue"    # ...then break it: SA and NSW now clash

print("variables:", variables)
print("good assignment satisfied?", satisfied(good))
print("bad  assignment satisfied?", satisfied(bad))

## Visualize the data & results

_Coloring the map of Australia, do these two colorings satisfy every 'neighboring states differ' rule?_

We'll count how many border rules each coloring breaks and chart the two counts. We do it in three steps: (1) the constraints and a violation counter, (2) the two colorings, (3) the bar chart.

### Step 1 — List the borders and count violations

Instead of a yes/no test, here we *count* how many borders are violated. The counter adds 1 for every edge whose two states share a color, so a fully valid coloring scores 0.

In [ ]:
borders = [
    ("WA", "NT"), ("WA", "SA"), ("NT", "SA"), ("NT", "Q"),
    ("SA", "Q"), ("SA", "NSW"), ("SA", "V"), ("Q", "NSW"), ("NSW", "V"),
]

def violations(c):
    # Count borders whose two endpoints share the same color.
    return sum(1 for a, b in borders if c[a] == c[b])

### Step 2 — Build the good and broken colorings

Same idea as before, but this version also colors `T` (Tasmania). We copy the good coloring and recolor `NSW` to blue so it clashes with `SA`, giving exactly one violation.

In [ ]:
good = {"WA": "red", "NT": "green", "SA": "blue",
        "Q": "red", "NSW": "green", "V": "red", "T": "red"}

bad = dict(good)        # copy, then break it
bad["NSW"] = "blue"    # NSW now clashes with SA

vals = [violations(good), violations(bad)]

### Step 3 — Chart the violation counts

The good coloring should be a zero-height bar (no rules broken) while the bad one rises to the single border it violates. The chart makes the difference between a satisfying and a non-satisfying assignment immediate.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["good", "bad"], vals, color=["#7ee787", "#ff7b72"])

# Annotate each bar with its violation count.
for i, v in enumerate(vals):
    ax.text(i, v, str(v), ha="center", va="bottom")

ax.set_title("Australia map coloring: violations (9 borders)")
plt.show()